In [ ]:
import os 
from quad_race_env import *
from randomization import *
from stable_baselines3 import PPO
from quadcopter_animation import animation
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_log(log_dir):
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    tags = event_acc.Tags()['scalars']
    
    # print(tags)
    # raise Exception('stop')
    log = {'step': [], 'ep_rew_mean': []}
    
    for scalar_event in event_acc.Scalars('rollout/ep_rew_mean'):
        log['step'].append(scalar_event.step)
        log['ep_rew_mean'].append(scalar_event.value)
        
    log = {key: np.array(values) for key, values in log.items()}
    # sub sample in steps of 1e6
    indices = log['step'] % 1e6 == 0
    log = {key: values[indices] for key, values in log.items()}
    # get the highest value and corresponding step
    max_index = np.argmax(log['ep_rew_mean'])
    log['max_ep_rew_mean'] = log['ep_rew_mean'][max_index]
    log['max_step'] = log['step'][max_index]
    return log
    
def get_best_model(log_dir):
    log = get_log(log_dir)
    # print('highest mean episode reward:', log['max_ep_rew_mean'], 'at step:', log['max_step'])
    model_dir = log_dir
    model_dir = model_dir.replace('logs', 'models')
    model_dir = model_dir[:-2]
    path_best_model = model_dir + f'/{log["max_step"]}.zip'
    print('best model:', path_best_model, 'mean_ep_rew:', log['max_ep_rew_mean'])
    return path_best_model, log['max_ep_rew_mean']

env = Quadcopter3DGates(
    num_envs=1,
    randomization=randomization_dummy_30_percent,
    initialize_at_random_gates=False,
    initialize_on_ground=True,
    pause_if_collision=False,
)

# get latest model from model/ground_exp
def get_latest_model(path):
    models = os.listdir(path)
    # all files end with {number}.zip, get the highest number
    models = sorted(models, key=lambda x: int(x.split('.')[0].split('_')[-1]))
    print(models[-1])
    model = PPO.load(path + '/' + models[-1])
    return model

#  ANIMATION FUNCTION
action_list = []
reward_list = []
def animate_policy(model, env, **kwargs):
    env.reset()
    def run():
        actions, _ = model.predict(env.states, deterministic=False)
        action_list.append(actions)
        states, rewards, dones, infos = env.step(actions)
        reward_list.append(rewards)
        print('speed = ', np.mean(np.linalg.norm(states[:, 3:6], axis=1)))
        print('crashed = ', np.sum(dones))
        rate = states[9:12]
        # if any rate is higher 1800 pritn
        if np.any(np.abs(rate) > 1800*np.pi/180):
            print(rate)
            print('rate too high')
        # print(infos)
        return env.render()
    animation.view(run, gate_pos=env.gate_pos, gate_yaw=env.gate_yaw, reset_func=env.reset, **kwargs)

# model, _ = get_best_model('logs/ground_exp/test1_0')
# model, _ = get_best_model('logs/perception_exp/test2_0')
# model, _ = get_best_model('logs/perception_exp/long_oval_good_axis_convention_0')
model = 'models/perception_exp/long_oval_good_axis_convention/100000000'
# model = 'models/perception_exp/long_oval_good_axis_convention_from_ground/100000000'
# model, _ = get_best_model('logs/perception_exp/long_oval_good_axis_convention_from_ground_0')
model = PPO.load(model)
animate_policy(model, env)

In [2]:
# best model: models/perception_exp/circle_small_1/22000000.zip mean_ep_rew: 118.86217498779297
# best model: models/perception_exp/circle_1/290000000.zip mean_ep_rew: 141.7411651611328